<a href="https://colab.research.google.com/github/Prabhu12345678/Generative_AI_Skill_Level_2/blob/main/tf_idf_Solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [163]:
# Manages on browser file loading,
# not required when run locally

#from google.colab import files

#uploaded = files.upload()

In [164]:
import pandas as pd
import networkx as nx

In [165]:
# Load the data

df = pd.read_csv('./train.csv')

df.info()

print('No of Rows missing the keyword', df['keyword'].isnull().sum())
print('No of Rows missing the location', df['location'].isnull().sum())

df = df.drop('keyword', axis=1)
df = df.drop('location',axis=1)

display(df.head(5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          20000 non-null  int64 
 1   keyword     15969 non-null  object
 2   location    15994 non-null  object
 3   text        20000 non-null  object
 4   target      20000 non-null  object
 5   created_at  20000 non-null  object
dtypes: int64(1), object(5)
memory usage: 937.6+ KB
No of Rows missing the keyword 4031
No of Rows missing the location 4006


,id,text,target,created_at
0,1,=> 💨💥🛑 Category 3 hurricane making landfall ne...,weather,2026-06-10 06:03:48
1,2,"!!! 👇🔥⚠️ Record-breaking heatwave in Mumbai, I...",weather,2026-06-27 19:15:12
2,3,?? Severe drought causing water rationing acro...,weather,2026-06-17 19:14:11
3,4,"🛑🔥 Tornado warning issued for Toronto, Canada ...",weather,2026-06-13 00:08:24
4,5,😱 Building structural collapse reported in Ber...,incident,2026-06-10 01:29:10


In [166]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from bs4 import BeautifulSoup

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

lemmatizer = WordNetLemmatizer()

def clean_text(text):

  # Remove URLs
  text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+','',text)

  # Remove HTML Tags
  text = BeautifulSoup(text, 'html.parser').get_text()

  # Remove emojis
  emoji_pattern = re.compile(
      "["
      "\U0001F600-\U0001F64F"      #Emotions
      "\U0001F300-\U0001F5FF"      #Symbols & Pictographs
      "\U0001F680-\U0001F6FF"      #Transport & Map Symbols
      "\U0001F1E0-\U0001F1FF"      #Flags (iOS)
      "\U00002702-\U000027B0"
      "\U000024C2-\U0001F251"
      "]+", flags=re.UNICODE)
  text = emoji_pattern.sub(r'', text)

  # Remove Punctuation
  text = re.sub(r'[^\w\s]', '', text)

  # Text to lower
  text = text.lower()

  # Remove non-alphanumeric characters
  text = re.sub(r'[^a-zA-Z0-9\s]','', text)

  # Remove numbers
  text = re.sub(r'\d+', '', text)

  # Tokenize the text
  words = nltk.word_tokenize(text)

  # Remove stopwords
  stop_words = set(stopwords.words('english'))

  cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words and word.strip() != '']

  # Extra space between quotes is required
  return ' '.join(cleaned_words)

df['text'] = df['text'].apply(clean_text)

clean_df = df.head(5)

display(clean_df)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,id,text,target,created_at
0,1,category hurricane making landfall near tokyo ...,weather,2026-06-10 06:03:48
1,2,recordbreaking heatwave mumbai india temperatu...,weather,2026-06-27 19:15:12
2,3,severe drought causing water rationing across ...,weather,2026-06-17 19:14:11
3,4,tornado warning issued toronto canada surround...,weather,2026-06-13 00:08:24
4,5,building structural collapse reported berlin g...,incident,2026-06-10 01:29:10


In [167]:
from sklearn.feature_extraction.text import CountVectorizer

sample_corpora = df['text'].head(2)

#sample_corpora = ['love India','love USA']
#sample_corpora = ['likes to have a masala dosa and coffee','prefers ice cream after the dosa']

count_vectorizer = CountVectorizer()

wm = count_vectorizer.fit_transform(sample_corpora)

# Check the vocabulary to see how words are ordered
print(count_vectorizer.vocabulary_)

{'category': 0, 'hurricane': 2, 'making': 6, 'landfall': 5, 'near': 8, 'tokyo': 12, 'japan': 4, 'tonight': 13, 'recordbreaking': 10, 'heatwave': 1, 'mumbai': 7, 'india': 3, 'temperature': 11, 'reaching': 9}


In [168]:
doc_names = ['Doc0','Doc1']

feat_names = count_vectorizer.get_feature_names_out()

sample_df = pd.DataFrame(wm.toarray(), index=doc_names, columns=feat_names)

display(sample_df)


,category,heatwave,hurricane,india,japan,landfall,making,mumbai,near,reaching,recordbreaking,temperature,tokyo,tonight
Doc0,1,0,1,0,1,1,1,0,1,0,0,0,1,1
Doc1,0,1,0,1,0,0,0,1,0,1,1,1,0,0


In [169]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()

wm = tfidf_vectorizer.fit_transform(sample_corpora)

feat_names = tfidf_vectorizer.get_feature_names_out()

sample_df = pd.DataFrame(wm.toarray(), index=doc_names, columns=feat_names)

display(sample_df)

,category,heatwave,hurricane,india,japan,landfall,making,mumbai,near,reaching,recordbreaking,temperature,tokyo,tonight
Doc0,0.353553,0.000000,0.353553,0.000000,0.353553,0.353553,0.353553,0.000000,0.353553,0.000000,0.000000,0.000000,0.353553,0.353553
Doc1,0.000000,0.408248,0.000000,0.408248,0.000000,0.000000,0.000000,0.408248,0.000000,0.408248,0.408248,0.408248,0.000000,0.000000


In [170]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
# Parameters : train_test_split(Data, Labels), text is the data, & target is the lables
# Splitter outputs : X refers to data, y refers to lables
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['target'], test_size=0.2, random_state=123)

tfidf_vectorizer = TfidfVectorizer()

#display(X_train.head(2))
#display(y_train.head(2))
#display(X_test.head(2))
#display(y_test.head(2))

tfidf_train_vector = tfidf_vectorizer.fit_transform(X_train, y_train)
tfidf_test_vector = tfidf_vectorizer.transform(X_test)

print('Training vector shape :', tfidf_train_vector.shape)
print('Test vector shape :', tfidf_test_vector.shape)


Training vector shape : (16000, 122)
Test vector shape : (4000, 122)


In [171]:
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

classifier = RandomForestClassifier(n_estimators=100, random_state=123)

# Split the data into training and testing sets
X_ran_train, X_ran_test, y_ran_train, y_ran_test = train_test_split(tfidf_train_vector, y_train, test_size=0.2, random_state=123)

classifier.fit(X_ran_train, y_ran_train)

y_pred = classifier.predict(X_ran_test)

# Average (optional) : weighted or micro or macro or None, but None will o/p in [1.0, 1.0, 1.0]
f1score = f1_score(y_ran_test, y_pred, average='weighted')

print('F1 Score : \n', f1score)

# 'labels=' optional in confusion matrix
cnf_matrix = confusion_matrix(y_ran_test, y_pred, labels=np.unique(y_pred))

print('Confusion Matrix : \n', cnf_matrix)

classification_report = classification_report(y_ran_test, y_pred)

print('Classification Report : \n', classification_report)

#

F1 Score : 
 1.0
Confusion Matrix : 
 [[1036    0    0]
 [   0 1097    0]
 [   0    0 1067]]
Classification Report : 
               precision    recall  f1-score   support

    disaster       1.00      1.00      1.00      1036
    incident       1.00      1.00      1.00      1097
     weather       1.00      1.00      1.00      1067

    accuracy                           1.00      3200
   macro avg       1.00      1.00      1.00      3200
weighted avg       1.00      1.00      1.00      3200

